## Climate scenarios

Take a look at [the climate scenarios in our world in data](https://ourworldindata.org/explorers/ipcc-scenarios) to get a feel for climate projections going forwards. The data needed for the cell below is downloaded and can be found in `co2-emissions-ssp12345-baseline-all.csv`.

Here we downloaded one set of data that we can use in the notebook which is total (not per capita) global CO2 emissions in the different SSP scenarios at "baseline" RCP:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import csv

# The data comes in a strange format with many timeseries appearing as
# different rows in the CSV file. We need to reorganize it into a more
# convenient format. This function should work for any of the datasets
# downloaded from
#
# https://ourworldindata.org/explorers/ipcc-scenarios

def read_owid_csv(filename: str, column: str) -> dict[str, tuple[list[float], list[float]]]:
    """Parse the our world in data CSV file"""
    emissions_unordered = {}

    with open(filename) as fin:
        for row in csv.DictReader(fin):
            scenario = row['Entity']
            year = float(row['Year'])
            datapoint = row[column]
            if not datapoint:
                continue
            value = float(datapoint) / 1e9 # Convert to GtCO2
            if scenario not in emissions_unordered:
                emissions_unordered[scenario] = []
            emissions_unordered[scenario].append((year, value))

    emissions = {}

    for scenario, data in emissions_unordered.items():
        years, values = zip(*sorted(data))
        emissions[scenario] = (years, values)

    return emissions


# csv_file = 'co2-emissions-ssp12345-baseline-all.csv'
csv_file = 'global-carbon-dioxide-emissions.csv'
column = 'CO2 emissions - Region: global'

future_emissions = read_owid_csv(csv_file, column)

# Show the available scenarios:
for scenario, (years, values) in future_emissions.items():
    print(scenario)

In [ ]:
# Now plot some scenarios:

# Now we can plot the data. We will show only the baseline scenarios (RCP8.5)

scenarios_to_plot = [
    'SSP1 - Baseline',
    'SSP2 - Baseline',
    'SSP3 - Baseline',
    'SSP4 - Baseline',
    'SSP5 - Baseline',
]
# scenarios_to_plot = [s for s in emissions.keys() if 'Baseline' in s]

fig = plt.figure()
ax = fig.add_subplot(111)

for scenario in scenarios_to_plot:
    years, values = future_emissions[scenario]
    ax.plot(years, values, label=scenario)

ax.set_xlabel('Year')
ax.set_ylabel('CO2 emissions (GtCO2)')
ax.set_ylim(0, 140)
ax.set_xlim(2005, 2100)
ax.legend()
plt.show()

## Data from the past for CO2 emissions and atmospheric levels

This is from a previous notebook but let's load and look at the data. This needs `co2-world.csv` and `co2_mm_mlo.csv`.

In [ ]:
# CO2 emissions from fossil fuels from:
#
# https://ourworldindata.org/co2-emissions

import matplotlib.pyplot as plt
import numpy as np
import csv

with open('co2-world.csv') as fin:

    past_co2_emissions_years = []
    past_co2_emissions = []

    for row in csv.DictReader(fin):
        year = int(row['Year'])
        value = float(row['Annual CO2 emissions']) / 1e9 # Convert to GtCO2
        past_co2_emissions_years.append(year)
        past_co2_emissions.append(value)

plt.plot(past_co2_emissions_years, past_co2_emissions)
plt.xlabel('Year')
plt.ylabel('CO2 emissions (GtCO2)')
plt.title('CO2 emissions from fossil fuels')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import csv

# https://climate.nasa.gov/vital-signs/carbon-dioxide/?intent=121

with open('co2_mm_mlo.csv') as fin:

    past_co2_atmospheric_years = []
    past_co2_atmospheric = []

    for row in csv.DictReader(fin):
        year = float(row['decimal date'])
        value = float(row['average']) # ppm
        past_co2_atmospheric_years.append(year)
        past_co2_atmospheric.append(value)

plt.plot(past_co2_atmospheric_years, past_co2_atmospheric)
plt.xlabel('Year')
plt.ylabel('CO2 concentration (ppm)')
plt.title('Atmospheric CO2 concentration over time')
plt.show()

## Relate emissions to concentration

The conversion from Gt (gigatonnes) to ppm (parts per million) for atmospheric CO2 is:
$$
1 \text{ppm} = 7.82 \text{Gt}
$$
We can use this to put **cumulative** emissions and constration levels on the same scale:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Cumulative emissions in units of ppm:
cumulative_co2 = np.cumsum(past_co2_emissions) / 7.82

ax = plt.subplot(111)
ax.plot(past_co2_emissions_years, cumulative_co2, label='cumulative emissions')
ax.plot(past_co2_atmospheric_years, past_co2_atmospheric, label='concentration')
ax.legend()
ax.set_xlabel('Year')
ax.set_ylabel('ppm')

## Line up emissions and concentration

The concentration doesn't start at zero so we should count our cumulative emissions relative to an older concentration value. We need to add about 300ppm to the cumulative emissions so that they line up at 1950.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cumulative_co2_offset1 = cumulative_co2 + 280

ax = plt.subplot(111)
ax.plot(past_co2_emissions_years, cumulative_co2_offset1, label='cumulative emissions (offset)')
ax.plot(past_co2_atmospheric_years, past_co2_atmospheric, label='concentration')
ax.legend()
ax.set_xlabel('Year')
ax.set_ylabel('ppm')

## Emissions and concentrations continued...

The cumulative emissions still don't line up. This is because about half of the CO2 emitted is absorbed by other things. On this timescale most of this is absorbed into the upper ocean: as CO2 increases in the atmosphere it also increases in the upper ocean.

We can account for this to make the curves line up:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# This is very approximate. Can be improved:
co2_loss = 0.5 # CO2 absorbed by e.g. oceans
cumulative_co2_offset2 = 300 + cumulative_co2 * (1 - co2_loss)

ax = plt.subplot(111)
ax.plot(past_co2_emissions_years, cumulative_co2_offset2, label='cumulative emissions (offset)')
ax.plot(past_co2_atmospheric_years, past_co2_atmospheric, label='concentration')
ax.legend()
ax.set_xlabel('Year')
ax.set_ylabel('ppm')

## CO2 removal

We have a pretty good lineup between emissions and concentrations. The model is simple: we assume that half the extra CO2 in the atmosphere goes other places (e.g. ocean) and otherwise that the conctration increases from 300ppm in 1750 according to cumulative emissions.

Over a longer timescale the CO2 will get removed from the atmosphere. In the first notebook we propose the following ODE to describe how this evolves:
$$
\frac{d \rho}{dt} = \kappa E - \frac{1}{t_{CO2}}(\rho - \rho_{1750})
$$
Here $E$ is the rate of emissions. The timescale $t_{CO2}$ is suggested to be about 100 years. Generally CO2 is expected to last for about 200 years in the atmosphere but that only loosely fixes the parameter $t_{CO2}$. Otherwise $\rho$ is the concentration varying over time with $\rho_{1750}$ being the 1750 value of the concentration. The parameter $\kappa$ represents the fact that about half of the CO2 goes into the oceans immediately.

We can tune the parameters of this model to get a reasonable fit to the data. Playing with the parameters it seems that $\kappa = 0.75$ gives a good fit:

In [ ]:
# Compare the ODE with the data

import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import numpy as np

kappa = 0.75 # T
t_co2 = 100 # years
rho_1750 = 290 # ppm

# Convert GtCO2 to ppm (1 GtCO2 = 7.82 ppm) so ppm/year
past_co2_emissions_ppm = [value / 7.82 for value in past_co2_emissions]

# Emissions function (in ppm/year) from the data
def E(t):
    return np.interp(t, past_co2_emissions_years, past_co2_emissions_ppm)

def f_rho_rhs(t, rho):
    return kappa * E(t) - (1/t_co2) * (rho - rho_1750)

t_span = (1750, 2020)
t_eval = np.linspace(*t_span, 1000)

sol = solve_ivp(f_rho_rhs, t_span, [rho_1750], t_eval=t_eval)

plt.plot(sol.t, sol.y[0], label='Model')
plt.plot(past_co2_atmospheric_years, past_co2_atmospheric, label='Data')
plt.xlabel('Year')
plt.ylabel('CO2 concentration (ppm)')
plt.title('Atmospheric CO2 concentration over time')
plt.legend()
plt.show()


## Projecting into the future

If we have a model for how atmospheric CO2 levels evolve over time as a function of emissions then we can use it to project forwards.

Let's try two simple scenarios:

- **net zero**: suppose that CO2 emissions drop to zero (net zero) in 2025.
- **constant**: suppose that CO2 emissions stop increasing and stay constant at their current level.

We adjust our function $E(t)$ accordingly:

In [ ]:
# Compare the ODE with the data

import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import numpy as np

kappa = 0.75
t_co2 = 100 # years
rho_1750 = 290 # ppm

# Convert GtCO2 to ppm (1 GtCO2 = 7.82 ppm) so ppm/year
past_co2_emissions_ppm = [value / 7.82 for value in past_co2_emissions]

# Emissions function (in ppm/year) from the data
def E_net_zero(t):
    if t > 2025:
        return 0 # Net Zero!
    return np.interp(t, past_co2_emissions_years, past_co2_emissions_ppm)

def E_constant(t):
    if t > 2025:
        return past_co2_emissions_ppm[-1] # most recent value
    return np.interp(t, past_co2_emissions_years, past_co2_emissions_ppm)

def f_rho_rhs(t, rho, scenario):
    if scenario == 1:
        e = E_net_zero(t)
    elif scenario == 2:
        e = E_constant(t)
    return kappa * e - (1/t_co2) * (rho - rho_1750)

t_span = (1750, 2200)
t_eval = np.linspace(*t_span, 1000)

sol1 = solve_ivp(f_rho_rhs, t_span, [rho_1750], t_eval=t_eval, args=(1,))
sol2 = solve_ivp(f_rho_rhs, t_span, [rho_1750], t_eval=t_eval, args=(2,))

plt.plot(sol1.t, sol1.y[0], label='Net Zero')
plt.plot(sol2.t, sol2.y[0], label='Constant')
plt.plot(past_co2_atmospheric_years, past_co2_atmospheric, label='Data')
plt.xlabel('Year')
plt.ylabel('CO2 concentration (ppm)')
plt.title('Atmospheric CO2 concentration over time')
plt.legend()
plt.show()

## Removal of CO2

It takes a long time for atmospheric CO2 levels to go down. Even if we achieve net zero immediately it takes 200 years for atmospheric CO2 to get close to preindustrial levels. This obviously depends a lot on our arbitrarily chosen parameter $t_{CO2}$ which was set to 100 years. How does this compare with the projections from more complex models for the different scenarios shown at our [world in data](https://ourworldindata.org/explorers/ipcc-scenarios?Metric=Greenhouse+gas+concentrations&Rate=Per+capita&Region=Global&country=SSP1+-+Baseline~SSP2+-+Baseline~SSP3+-+Baseline~SSP4+-+Baseline~SSP5+-+Baseline)?

Note that this model ignores any efect of tipping points. With tipping points we do not necessarily expect CO2 levels to return to preindustrial levels.

## Temperature levels

We can also recover the data for temperature levels from the previous notebook. This requires `temperature.txt` from NASA:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import csv

# https://climate.nasa.gov/vital-signs/global-temperature/?intent=12
with open('temperature.txt', 'r') as f:
    lines = f.readlines()

lines = lines[5:] # skip header lines

temperature_years = []
temperature_raw = []
temperature_smooth = []

for line in lines:
    year, raw, smooth = line.split()
    temperature_years.append(int(year))
    temperature_raw.append(float(raw))
    temperature_smooth.append(float(smooth))

plt.plot(temperature_years, temperature_raw, label='Raw')
plt.plot(temperature_years, temperature_smooth, label='Smooth')
plt.xlabel('Year')
plt.ylabel('Temperature offset (C)')
plt.legend()
plt.show()

## Temperature model

Our basic temperature model is
$$
C \frac{dT}{dt} = (1 - \alpha) R_{in} - (1 - (\beta + \beta^\star)) \sigma T^4
$$
and we suppose that $\beta^\star = G (\rho - \rho_{1750})$ which relates the greenhouse effect to CO2 concentration in the atmosphere. After fiddling with the parameters a bit we can match the data reasonably well to see how global temperature evolves as atmospheric concentrations change:

In [ ]:
from scipy.integrate import solve_ivp

rho_1750 = 290 # ppm
sigma = 5.67e-8 # W/m^2/K^4
alpha = 0.3 # albedo
R = 340 # W/m^2

# These parameters are chosen partly based on fiddling around
# to see what matches the data but also thinking about what
# should happen for particular values of the dynamic variables.

C = 1 # heat capacity of ??? in J/m^2/K
beta = 0.386 # baseline greenhouse effect
G = 10e-5 # greenhouse effect coefficient in 1/ppm

def beta_star(t):
    rho = np.interp(t, past_co2_atmospheric_years, past_co2_atmospheric)
    return G * (rho - rho_1750)

def f_T_rhs(t, T):
    return (1/C) * ((1 - alpha) * R - (1 - beta - beta_star(t)) * sigma * T**4)

t_span = (1950, 2020)
t = np.linspace(*t_span, 1000)

T0 = 15 + 273 # K (average initial surface temp)

sol = solve_ivp(f_T_rhs, t_span, [T0], t_eval=t, rtol=1e-5)

plt.plot(sol.t, sol.y[0] - 273 - 15, label='model')
plt.plot(temperature_years, temperature_raw, label='raw data')
plt.xlabel('Year')
plt.ylabel('Temperature increase (degC)')
# plt.xlim(1950, 2020)
plt.legend()
plt.show()

## Closing the loop with a full model

Okay so we now have a complete model. This is the atmospheric concentration ODE:
$$
\frac{d \rho}{dt} = \kappa E(t) - \frac{1}{t_{CO2}}(\rho - \rho_{1750})
$$
This is the temperature ODE:
$$
C \frac{dT}{dt} = (1 - \alpha) R_{in} - (1 - (\beta + \beta^\star)) \sigma T^4
$$
Where $\beta^\star = G(\rho - \rho_{1750})$.

These are the parameters/constants:

- $\kappa = 0.75$ fraction of CO2 not absorbed by the oceans.
- $t_{CO2} = 100\,\text{years}$. Timescale for CO2 to be removed from atmosphere.
- $\rho_{1750} = 290\,\text{ppm}$. Preindustrial concetration of atmospheric CO2.
- $C = 1$ (J/m$^2$/K) heat capacity of the surface of the Earth
- $\alpha=0.3$ albedo of the Earth
- $\beta=0.386$ baseline greenhouse effect parameter.
- $\sigma = 5.67\times 10^{-8}$ W/m$^2$/K$^4$ Stefan-Boltzmann constant.
- $G = 10^{-5}$ 1/ppm greenhouse effect sensitivity parameter.

The function $E(t)$ represents emissions of CO2 over time and is our "scenario" like the SSP/RCP scenarios. In fact we can now use those scenarios in the model since we have the emissions data from them:

In [ ]:
from scipy.integrate import solve_ivp

kappa = 0.75
t_co2 = 100 # years
rho_1750 = 290 # ppm
sigma = 5.67e-8 # W/m^2/K^4
alpha = 0.3 # albedo
beta = 0.386 # baseline greenhouse effect
C = 1 # heat capacity of the atmosphere in J/m^2/K
G = 10e-5
R = 340 # W/m^2

def E(t, scenario):
    if t < 2025:
        return np.interp(t, past_co2_emissions_years, past_co2_emissions) / 7.8
    elif scenario == 'Net Zero':
        return 0
    elif scenario == 'Constant':
        return 5.2 # ppm/year
    else:
        years, values = future_emissions[scenario]
        return np.interp(t, years, values) / 7.8 # convert GtCO2 to ppm

def f_T_rho_rhs(t, y, scenario):
    T, rho = y
    drho_dt = kappa * E(t, scenario) - (rho - rho_1750) / t_co2
    beta_star = G * (rho - rho_1750)
    dT_dt = (1/C) * ((1 - alpha) * R - (1 - beta - beta_star) * sigma * T**4)
    return [dT_dt, drho_dt]

t_span = (2000, 2100)
t = np.linspace(*t_span, 1000)

# Initial conditions at t = 2000
T0 = 15.5 + 273 # degC (average initial surface temp)
rho0 = 370


scenarios = [
    'Net Zero',
    'Constant',
    'SSP1 - 2.6',
    'SSP5 - Baseline',
]

for scenario in scenarios:
    sol = solve_ivp(f_T_rho_rhs, t_span, [T0, rho0], t_eval=t, args=(scenario,), rtol=1e-10)
    plt.plot(sol.t, sol.y[0] - 273 - 15, label=scenario)

plt.plot(temperature_years, temperature_raw, label='raw data', marker='x', linestyle='')
plt.xlabel('Year')
plt.ylabel('Temperature increase (degC)')
plt.xlim(*t_span)
plt.ylim(0, 4)
plt.legend()
plt.grid()
plt.show()



## Exercise

Try different scenarios in the model. Compare with the projections at our world in data.